# LLM Inference Optimization: T4-Compatible Pipeline (v2)

This is a clean rerun plan for the inference-optimization assessment. It keeps the previous notebook and artifacts untouched, writes new outputs to `/kaggle/working/llm-inference-artifacts-v2/`, and records failures instead of silently treating them as successes.

**Target:** `Qwen/Qwen3-4B` in FP16, quantized to symmetric AWQ W4A16; **draft:** `Qwen/Qwen3-0.6B`. These openly available same-family models avoid gated Llama access and provide a more meaningful target/draft size difference than the previous 1.7B target.

Key changes from v1:
- Quantization/evaluation runs on one T4 (`CUDA_VISIBLE_DEVICES=0`), avoiding `device_map="auto"` multi-GPU placement errors.
- AWQ uses symmetric `W4A16`, not `W4A16_ASYM`; current LLM Compressor guidance lists W4A16 for Turing GPUs (compute capability 7.5).
- Fresh artifact directories prevent stale safetensors from inflating the measured checkpoint size.
- vLLM is isolated and pinned to `0.31.0`; the previous `0.11.2` build rejected draft-model speculation.
- The suite measures 30 concurrent requests and compares sequential Transformers with vLLM. It tests draft-model speedup and output equality separately.

Perplexity uses the complete filtered WikiText-2 raw test split, with identical tokenization and 512-token evaluation windows for FP16 and AWQ. The tests report measured results and do not manufacture a pass if an accuracy or speed target is missed.

References: [LLM Compressor scheme guidance](https://docs.vllm.ai/projects/llm-compressor/en/latest/steps/choosing-scheme/), [vLLM quantization support](https://docs.vllm.ai/en/latest/features/quantization/index.html), [vLLM speculative decoding](https://docs.vllm.ai/en/latest/features/speculative_decoding/).

## 1. Check the Kaggle runtime

Use a fresh Kaggle notebook with Internet enabled and the Tesla T4 x2 accelerator. Run cells from top to bottom; this notebook writes only to `/kaggle/working/llm-inference-artifacts-v2/` and does not overwrite the v1 artifacts. This notebook intentionally uses one T4 for quantization and model serving because Qwen3-4B plus its 0.6B draft can fit within one T4 with the configured context and concurrency limits. The second T4 remains available rather than being automatically sharded by Transformers.

In [ ]:
!nvidia-smi

## 2. Create an isolated quantization environment

This keeps the notebook's default Python packages separate from the quantization stack. The model and reports use a new `v2` directory.

In [ ]:
%pip install -q uv
!uv python install 3.12
!if [ ! -x /kaggle/working/.venv-quant-v2/bin/python ]; then uv venv --python 3.12 --seed /kaggle/working/.venv-quant-v2; fi
!uv pip install --python /kaggle/working/.venv-quant-v2/bin/python --torch-backend=cu128 torch
!uv pip install --python /kaggle/working/.venv-quant-v2/bin/python "transformers==4.57.6" "llmcompressor==0.11.0" datasets accelerate safetensors pyyaml huggingface_hub

## 3. Define a shared load-test prompt set

The sequential Transformers baseline and vLLM concurrent test use the same 30 prompts and the same 32-token generation limit.

In [ ]:
from pathlib import Path
import json

ROOT = Path("/kaggle/working/llm-inference-artifacts-v2")
REPORT_DIR = ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

prompt_templates = [
    "Explain how post-training quantization reduces LLM memory and describe one accuracy trade-off.",
    "Describe how self-attention uses query, key, and value tensors to process a sequence.",
    "Compare request latency, time to first token, inter-token latency, and aggregate throughput.",
    "Explain how a key-value cache reduces repeated computation during autoregressive generation.",
    "Describe how continuous batching and paged KV-cache allocation improve serving utilization.",
    "Explain how calibration samples affect activation-aware weight quantization and model quality.",
]
followups = [
    "Use a small numerical example.",
    "Keep the answer practical and suitable for an inference engineer.",
    "Include one limitation and one method to measure it.",
    "Explain the trade-off without assuming specialized hardware.",
    "Compare the behavior during single-user and concurrent serving.",
]
load_prompts = [
    f"{prompt_templates[i % len(prompt_templates)]} {followups[(i // len(prompt_templates) + i) % len(followups)]} Request {i + 1}."
    for i in range(30)
]
PROMPTS_PATH = REPORT_DIR / "load_prompts.json"
PROMPTS_PATH.write_text(json.dumps(load_prompts, indent=2), encoding="utf-8")
print(f"Saved {len(load_prompts)} shared prompts to {PROMPTS_PATH}")

## 4. Write the quantization, quality-evaluation, and Transformers baseline script

Important changes: use FP16 weights and place the complete model on visible GPU 0. Perplexity is calculated on the same contiguous WikiText-2 token subset for both models. The script also measures a simple sequential 30-request Transformers baseline.

In [ ]:
%%writefile /tmp/run_quantization_pipeline_v2.py
from pathlib import Path
import csv
import gc
import json
import math
import os
import shutil
import subprocess
import time
import traceback

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.transform.awq import AWQModifier
from llmcompressor.modifiers.quantization import QuantizationModifier

ROOT = Path("/kaggle/working/llm-inference-artifacts-v2")
REPORT_DIR = ROOT / "reports"
BASE_DIR = ROOT / "qwen3-4b-original"
AWQ_DIR = ROOT / "qwen3-4b-awq-w4a16-symmetric"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
JSON_PATH = REPORT_DIR / "results.json"
CSV_PATH = REPORT_DIR / "transformers_benchmarks.csv"
SUMMARY_PATH = REPORT_DIR / "summary.md"
GEN_PATH = REPORT_DIR / "sample_generations.txt"
PROMPTS_PATH = REPORT_DIR / "load_prompts.json"
MODEL_ID = "Qwen/Qwen3-4B"
CALIBRATION_SAMPLES = 64
CALIBRATION_LENGTH = 256
EVAL_TOKENS = None  # None means use the complete filtered WikiText-2 test split
EVAL_BLOCK_LENGTH = 512
GEN_TOKENS = 64
BENCH_REPEATS = 3


def directory_size_bytes(path: Path) -> int:
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())


def gpu_rows():
    try:
        p = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv,noheader,nounits"],
            check=True, capture_output=True, text=True, timeout=10,
        )
        return [line.strip() for line in p.stdout.splitlines() if line.strip()]
    except Exception as exc:
        return [f"not available: {exc!r}"]


def evaluate_perplexity(model, tokenizer, test_texts):
    model.eval()
    combined = "\n\n".join(t for t in test_texts if t.strip())
    ids = tokenizer(combined, return_tensors="pt", add_special_tokens=False)["input_ids"][0]
    if EVAL_TOKENS is not None:
        ids = ids[:EVAL_TOKENS]
    total_nll = 0.0
    total_tokens = 0
    for start in range(0, len(ids), EVAL_BLOCK_LENGTH):
        chunk = ids[start:start + EVAL_BLOCK_LENGTH]
        if chunk.numel() < 2:
            continue
        input_ids = chunk.unsqueeze(0).to("cuda:0")
        with torch.inference_mode():
            outputs = model(input_ids=input_ids, labels=input_ids)
        token_count = int(input_ids.shape[-1] - 1)
        total_nll += float(outputs.loss.float().cpu()) * token_count
        total_tokens += token_count
        del input_ids, outputs
    if total_tokens == 0:
        raise RuntimeError("No valid tokens were available for perplexity evaluation.")
    mean_nll = total_nll / total_tokens
    return {
        "perplexity": round(math.exp(min(mean_nll, 20)), 5),
        "mean_token_loss": round(mean_nll, 7),
        "tokens_evaluated": total_tokens,
        "evaluation_block_length": EVAL_BLOCK_LENGTH,
    }


def generation_benchmark(model, tokenizer, prompt, label):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
    with torch.inference_mode():
        _ = model.generate(**inputs, max_new_tokens=8, do_sample=False, use_cache=True)
    times = []
    counts = []
    sample = ""
    allocated_before = torch.cuda.memory_allocated(0)
    torch.cuda.reset_peak_memory_stats(0)
    for _ in range(BENCH_REPEATS):
        torch.cuda.synchronize(0)
        t0 = time.perf_counter()
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=GEN_TOKENS, do_sample=False, use_cache=True)
        torch.cuda.synchronize(0)
        times.append(time.perf_counter() - t0)
        counts.append(int(output.shape[-1] - inputs["input_ids"].shape[-1]))
        sample = tokenizer.decode(output[0], skip_special_tokens=True)
        del output
    elapsed = sum(times) / len(times)
    peak = torch.cuda.max_memory_allocated(0)
    return {
        "stage": label,
        "repeats": BENCH_REPEATS,
        "generated_tokens_average": round(sum(counts) / len(counts), 2),
        "latency_seconds_average": round(elapsed, 4),
        "tokens_per_second": round((sum(counts) / len(counts)) / elapsed, 3) if elapsed else None,
        "peak_allocated_gpu_gib": round(peak / (1024**3), 3),
        "peak_extra_over_loaded_model_gib": round(max(0, peak - allocated_before) / (1024**3), 3),
        "sample_output": sample,
    }


def sequential_load_benchmark(model, tokenizer, prompts):
    model.eval()
    total_tokens = 0
    request_times = []
    started = time.perf_counter()
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
        torch.cuda.synchronize(0)
        request_start = time.perf_counter()
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=32, do_sample=False, use_cache=True)
        torch.cuda.synchronize(0)
        request_times.append(time.perf_counter() - request_start)
        total_tokens += int(output.shape[-1] - inputs["input_ids"].shape[-1])
        del inputs, output
    wall = time.perf_counter() - started
    return {
        "status": "ok",
        "engine": "Transformers sequential requests (one request at a time)",
        "requests": len(prompts),
        "max_new_tokens_per_request": 32,
        "output_tokens_total": total_tokens,
        "wall_seconds": round(wall, 3),
        "aggregate_tokens_per_second": round(total_tokens / wall, 3) if wall else None,
        "mean_request_latency_seconds": round(sum(request_times) / len(request_times), 4) if request_times else None,
        "gpu_memory_after_test": gpu_rows(),
    }


def main():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not visible. Enable a Kaggle GPU and rerun.")
    if torch.cuda.device_count() != 1:
        print(f"Note: this process sees {torch.cuda.device_count()} GPUs; the notebook runner should set CUDA_VISIBLE_DEVICES=0.")
    torch.cuda.set_device(0)
    torch.backends.cuda.matmul.allow_tf32 = True
    if not PROMPTS_PATH.is_file():
        raise FileNotFoundError(f"Missing shared prompt file: {PROMPTS_PATH}")
    prompts = json.loads(PROMPTS_PATH.read_text(encoding="utf-8"))
    if not BASE_DIR.exists() or not (BASE_DIR / "config.json").is_file() or not list(BASE_DIR.glob("*.safetensors")):
        print(f"Downloading {MODEL_ID} ...")
        snapshot_download(
            repo_id=MODEL_ID,
            local_dir=str(BASE_DIR),
            allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "*.jinja", "*.py", "*.tiktoken", "merges.txt", "vocab.json"],
        )

    tokenizer = AutoTokenizer.from_pretrained(str(BASE_DIR), trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Loading target model in FP16 on a single T4 (no automatic multi-GPU sharding)...")
    model = AutoModelForCausalLM.from_pretrained(
        str(BASE_DIR),
        torch_dtype=torch.float16,
        device_map={"": 0},
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    model.eval()
    parameter_count = sum(p.numel() for p in model.parameters())
    results = {
        "model": MODEL_ID,
        "quantization": "AWQ symmetric W4A16; Linear layers; lm_head excluded",
        "quantization_scheme": "W4A16",
        "quantization_symmetric": True,
        "calibration_samples": CALIBRATION_SAMPLES,
        "calibration_max_sequence_length": CALIBRATION_LENGTH,
        "evaluation_token_limit": EVAL_TOKENS,
        "evaluation_block_length": EVAL_BLOCK_LENGTH,
        "benchmark_generation_tokens": GEN_TOKENS,
        "benchmark_repeats": BENCH_REPEATS,
        "torch_version": torch.__version__,
        "transformers_version": __import__("transformers").__version__,
        "llmcompressor_version": __import__("importlib.metadata", fromlist=["version"]).version("llmcompressor"),
        "gpu_name": torch.cuda.get_device_name(0),
        "gpu_compute_capability": list(torch.cuda.get_device_capability(0)),
        "parameter_count": parameter_count,
        "estimated_fp16_parameter_bytes_gib": round(parameter_count * 2 / (1024**3), 3),
        "original_model_size_gib": round(directory_size_bytes(BASE_DIR) / (1024**3), 3),
        "gpu_before_quantization": gpu_rows(),
        "notes": [
            "Quantization, Transformers baselines, and perplexity run on one visible GPU to avoid automatic multi-GPU device placement.",
            "Perplexity covers the full filtered WikiText-2 raw test split for baseline and quantized models; each independent block contains up to 512 tokens.",
            "Transformers baseline is sequential request handling, not an optimized continuous-batching server.",
            "Runtime memory estimates must be interpreted alongside vLLM startup logs and KV-cache metrics; nvidia-smi total usage alone cannot separate memory categories.",
        ],
    }
    if not list(BASE_DIR.glob("*.safetensors")):
        raise RuntimeError(f"No safetensors weights found in {BASE_DIR}")

    print("Loading WikiText-2 calibration/evaluation data...")
    train_ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    train_ds = train_ds.filter(lambda row: bool(row["text"].strip()))
    train_ds = train_ds.shuffle(seed=42).select(range(CALIBRATION_SAMPLES))
    test_ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    test_ds = test_ds.filter(lambda row: bool(row["text"].strip()))
    eval_texts = [row["text"] for row in test_ds]

    print("Evaluating FP16 perplexity...")
    results["original_perplexity"] = evaluate_perplexity(model, tokenizer, eval_texts)

    print("Running sequential Transformers load benchmark on the shared 30 prompts...")
    results["transformers_sequential_load_test"] = sequential_load_benchmark(model, tokenizer, prompts)

    print("Running FP16 generation benchmark...")
    results["transformers_generation_benchmarks"] = []
    results["transformers_generation_benchmarks"].append(
        generation_benchmark(model, tokenizer,
                          "Explain how quantization and KV caching affect autoregressive inference.",
                          "fp16_transformers")
    )

    print("Starting symmetric AWQ W4A16 calibration and quantization...")
    if AWQ_DIR.exists():
        shutil.rmtree(AWQ_DIR)
    allocated_before_quant = torch.cuda.memory_allocated(0)
    torch.cuda.reset_peak_memory_stats(0)
    quant_start = time.perf_counter()
    recipe = [
        AWQModifier(),
        QuantizationModifier(ignore=["lm_head"], scheme="W4A16", targets=["Linear"]),
    ]
    oneshot(
        model=model,
        tokenizer=tokenizer,
        dataset=train_ds,
        text_column="text",
        recipe=recipe,
        num_calibration_samples=CALIBRATION_SAMPLES,
        max_seq_length=CALIBRATION_LENGTH,
    )
    quant_elapsed = time.perf_counter() - quant_start
    peak = torch.cuda.max_memory_allocated(0)
    results["quantization_time_seconds"] = round(quant_elapsed, 2)
    results["quantization_peak_allocated_gpu_gib"] = round(peak / (1024**3), 3)
    results["quantization_extra_peak_over_loaded_model_gib"] = round(max(0, peak - allocated_before_quant) / (1024**3), 3)

    print("Saving a fresh compressed checkpoint...")
    AWQ_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(AWQ_DIR), save_compressed=True)
    tokenizer.save_pretrained(str(AWQ_DIR))
    if not list(AWQ_DIR.rglob("*.safetensors")):
        raise RuntimeError(f"No safetensors weights were saved under {AWQ_DIR}")

    quant_bytes = directory_size_bytes(AWQ_DIR)
    original_bytes = directory_size_bytes(BASE_DIR)
    results["quantized_model_size_gib"] = round(quant_bytes / (1024**3), 3)
    results["quantized_model_size_pass_le_5_5_gib"] = quant_bytes <= 5.5 * (1024**3)
    results["size_reduction_percent"] = round((original_bytes - quant_bytes) / original_bytes * 100, 2)
    results["compression_ratio"] = round(original_bytes / quant_bytes, 3) if quant_bytes else None
    saved_config = json.loads((AWQ_DIR / "config.json").read_text(encoding="utf-8"))
    qcfg = saved_config.get("quantization_config", {})
    results["saved_quantization_config"] = qcfg
    results["saved_quantization_method"] = qcfg.get("quant_method")

    # Finalize compressed-tensors dispatch after saving; the process sees only one GPU.
    from compressed_tensors.offload import dispatch_model
    dispatch_model(model)

    print("Evaluating quantized model with the same token subset...")
    try:
        results["awq_perplexity"] = evaluate_perplexity(model, tokenizer, eval_texts)
        p0 = results["original_perplexity"]["perplexity"]
        pq = results["awq_perplexity"]["perplexity"]
        results["perplexity_delta_percent"] = round((pq / p0 - 1) * 100, 3) if p0 else None
        results["perplexity_within_5_percent"] = abs(results["perplexity_delta_percent"]) <= 5 if results["perplexity_delta_percent"] is not None else False
    except Exception as exc:
        results["awq_perplexity"] = {"error": repr(exc), "traceback": traceback.format_exc()}
        results["perplexity_delta_percent"] = None
        results["perplexity_within_5_percent"] = False
        print("AWQ perplexity failed:", repr(exc))

    try:
        results["transformers_generation_benchmarks"].append(
            generation_benchmark(model, tokenizer,
                                 "Explain how quantization and KV caching affect autoregressive inference.",
                                 "awq_w4a16_transformers")
        )
    except Exception as exc:
        results["awq_generation_benchmark_error"] = {"error": repr(exc), "traceback": traceback.format_exc()}
        print("AWQ generation benchmark failed:", repr(exc))

    results["artifacts"] = {
        "original_checkpoint": str(BASE_DIR),
        "quantized_checkpoint": str(AWQ_DIR),
        "results_json": str(JSON_PATH),
        "benchmark_csv": str(CSV_PATH),
        "summary_markdown": str(SUMMARY_PATH),
        "load_prompts": str(PROMPTS_PATH),
    }
    results["gpu_after_quantization"] = gpu_rows()
    JSON_PATH.write_text(json.dumps(results, indent=2), encoding="utf-8")

    with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
        fields = ["stage", "repeats", "generated_tokens_average", "latency_seconds_average", "tokens_per_second", "peak_allocated_gpu_gib", "peak_extra_over_loaded_model_gib"]
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for row in results["transformers_generation_benchmarks"]:
            writer.writerow({k: row.get(k) for k in fields})

    lines = [
        "# LLM Inference Optimization - v2 Results", "",
        f"- Target model: `{MODEL_ID}`",
        "- Quantization: `AWQ symmetric W4A16` (Linear layers, `lm_head` excluded)",
        f"- Calibration: {CALIBRATION_SAMPLES} samples, max sequence length {CALIBRATION_LENGTH}",
        f"- Original checkpoint size: {results['original_model_size_gib']} GiB",
        f"- Quantized checkpoint size: {results['quantized_model_size_gib']} GiB",
        f"- Size reduction: {results['size_reduction_percent']}%",
        f"- Quantized checkpoint <= 5.5 GiB: `{results['quantized_model_size_pass_le_5_5_gib']}`",
        f"- FP16 WikiText-2 full-test perplexity: {results.get('original_perplexity', {}).get('perplexity', 'N/A')}",
        f"- AWQ WikiText-2 full-test perplexity: {results.get('awq_perplexity', {}).get('perplexity', 'N/A') if isinstance(results.get('awq_perplexity'), dict) else 'N/A'}",
        f"- Perplexity change: {results.get('perplexity_delta_percent', 'N/A')}%",
        f"- Perplexity within 5%: `{results.get('perplexity_within_5_percent')}`",
        f"- Sequential Transformers load throughput: {results['transformers_sequential_load_test'].get('aggregate_tokens_per_second')} tokens/s over {results['transformers_sequential_load_test']['requests']} requests",
        f"- Quantization time: {results.get('quantization_time_seconds', 'N/A')} s",
        f"- Quantization peak GPU allocation: {results.get('quantization_peak_allocated_gpu_gib', 'N/A')} GiB", "",
        "## Notes", "",
        "- Baseline and AWQ perplexity use the complete filtered WikiText-2 raw test split, tokenized together and evaluated in independent 512-token blocks.",
        "- The Transformers load test is sequential, while the vLLM test uses concurrent streaming requests.",
        "- The benchmark CSV and sample generations are available alongside this summary.",
    ]
    SUMMARY_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
    GEN_PATH.write_text("Baseline sample generation is stored in results.json.\n", encoding="utf-8")
    print("\nQUANTIZATION RESULTS")
    print(json.dumps(results, indent=2))
    print("\nReport:", JSON_PATH)


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        REPORT_DIR.mkdir(parents=True, exist_ok=True)
        failure = {"status": "failed", "error": repr(exc), "traceback": traceback.format_exc()}
        JSON_PATH.write_text(json.dumps(failure, indent=2), encoding="utf-8")
        print(json.dumps(failure, indent=2))
        raise

In [ ]:
import os
import subprocess
from pathlib import Path

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["TOKENIZERS_PARALLELISM"] = "false"
result = subprocess.run(
    ["/kaggle/working/.venv-quant-v2/bin/python", "/tmp/run_quantization_pipeline_v2.py"],
    env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print(result.stdout[-24000:])
print("Exit code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Quantization stage failed; inspect the traceback above before continuing.")

## 5. Download the draft model locally

The target and draft model share the Qwen3 family. The draft checkpoint is stored locally so vLLM doesn't need to resolve it during every server start.

In [ ]:
%%writefile /tmp/download_draft_v2.py
from pathlib import Path
from huggingface_hub import snapshot_download

root = Path("/kaggle/working/llm-inference-artifacts-v2")
draft_dir = root / "qwen3-0.6b-draft"
draft_dir.mkdir(parents=True, exist_ok=True)
snapshot_download(
    repo_id="Qwen/Qwen3-0.6B",
    local_dir=str(draft_dir),
    allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "*.jinja", "*.py", "*.tiktoken", "merges.txt", "vocab.json"],
)
print("Draft model saved to", draft_dir)

In [ ]:
import os
import subprocess

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
result = subprocess.run(
    ["/kaggle/working/.venv-quant-v2/bin/python", "/tmp/download_draft_v2.py"],
    env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print(result.stdout[-8000:])
if result.returncode != 0:
    raise RuntimeError("Draft model download failed; inspect the output above.")

## 6. Create a clean, current vLLM environment

The earlier `vLLM 0.11.2` did not initialize draft-model speculative decoding in this setup. This notebook uses `vLLM 0.31.0`, released in October 2026, in a separate Python 3.12 environment. Let `uv` select the matching current PyTorch/CUDA backend.

In [ ]:
!if [ ! -x /kaggle/working/.venv-vllm-v2/bin/python ]; then uv venv --python 3.12 --seed /kaggle/working/.venv-vllm-v2; fi
!uv pip install --python /kaggle/working/.venv-vllm-v2/bin/python --torch-backend=auto vllm==0.31.0 httpx

In [ ]:
import os
import subprocess

cmd = [
    "/kaggle/working/.venv-vllm-v2/bin/python", "-c",
    "import torch, vllm, transformers; print('vLLM:', vllm.__version__); print('Torch:', torch.__version__); print('Transformers:', transformers.__version__); print('CUDA:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'); print('Capability:', torch.cuda.get_device_capability(0) if torch.cuda.is_available() else 'none')",
]
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
result = subprocess.run(cmd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError("vLLM environment check failed.")

## 7. Write the vLLM benchmark runner

`load` mode submits 30 streaming requests concurrently and records TTFT, request latency, inter-chunk latency estimate, throughput, KV-cache metrics, and GPU memory. `spec` mode runs single-user requests with roughly 100-token prompts and 96-token completions, which is the relevant mode for measuring speculative decoding speedup per user.

In [ ]:
%%writefile /tmp/run_vllm_bench_v2.py
import argparse
import asyncio
import json
import os
import subprocess
import time
import traceback
from pathlib import Path

import httpx
from transformers import AutoTokenizer

parser = argparse.ArgumentParser()
parser.add_argument("--label", required=True)
parser.add_argument("--model", required=True)
parser.add_argument("--tokenizer", required=True)
parser.add_argument("--output", required=True)
parser.add_argument("--mode", choices=["load", "spec"], required=True)
parser.add_argument("--speculative", action="store_true")
parser.add_argument("--draft-model", default="")
parser.add_argument("--requests", type=int, default=30)
parser.add_argument("--port", type=int, default=8000)
args = parser.parse_args()

ROOT = Path("/kaggle/working/llm-inference-artifacts-v2")
REPORT_DIR = ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
BASE_URL = f"http://127.0.0.1:{args.port}"
VLLM_CLI = "/kaggle/working/.venv-vllm-v2/bin/vllm"
LOAD_PROMPTS_PATH = REPORT_DIR / "load_prompts.json"


def tail_text(path: Path, limit: int = 12000) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="replace")[-limit:]
    except Exception:
        return ""


def gpu_memory_rows():
    try:
        proc = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv,noheader,nounits"],
            check=True, capture_output=True, text=True, timeout=10,
        )
        return [line.strip() for line in proc.stdout.splitlines() if line.strip()]
    except Exception as exc:
        return [f"not available: {exc!r}"]


def percentile(values, q):
    if not values:
        return None
    vals = sorted(values)
    return round(vals[round((len(vals) - 1) * q)], 4)


def server_memory_lines(path: Path):
    needles = ("GPU KV cache size", "Model loading took", "model weights", "KV cache", "cache block", "Memory profiling", "Available KV cache memory")
    try:
        lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
        return [line[-700:] for line in lines if any(n.lower() in line.lower() for n in needles)][-40:]
    except Exception:
        return []


def build_spec_prompts(tokenizer):
    seeds = [
        "A production language model API receives many requests with different prompt lengths. Explain how request scheduling, token generation, and key-value caching affect latency and throughput. Compare a naive sequential implementation with an inference server that can batch work continuously, and describe how to measure the difference fairly.",
        "Post-training quantization reduces the number of bits used for model weights, but it can change the outputs. Explain the roles of calibration samples, group size, symmetric and asymmetric quantization, and perplexity evaluation. Include the main memory and accuracy trade-offs an engineer should consider before deployment.",
        "Speculative decoding uses a smaller draft model to propose candidate tokens that a larger target model verifies. Explain why the approach can be lossless when correctly implemented, why acceptance rate matters, and why a draft model may still make generation slower. Describe a controlled benchmark that compares target-only decoding with speculative decoding.",
    ]
    out = []
    for seed in seeds:
        ids = tokenizer(seed, add_special_tokens=False)["input_ids"]
        while len(ids) < 100:
            ids = ids + tokenizer(seed, add_special_tokens=False)["input_ids"]
        ids = ids[:100]
        prompt = tokenizer.decode(ids, skip_special_tokens=True)
        out.append({"prompt": prompt, "prompt_tokens": len(tokenizer(prompt, add_special_tokens=False)["input_ids"])})
    return out


async def stream_one(client, tokenizer, served_model, prompt, max_tokens, include_text=False):
    t0 = time.perf_counter()
    first_delta = None
    last_delta = None
    intervals = []
    pieces = []
    usage_tokens = None
    request_body = {
        "model": served_model,
        "prompt": prompt,
        "temperature": 0.0,
        "max_tokens": max_tokens,
        "stream": True,
        "stream_options": {"include_usage": True},
    }
    async with client.stream("POST", f"{BASE_URL}/v1/completions", json=request_body) as response:
        if response.status_code >= 400:
            body = await response.aread()
            raise RuntimeError(f"completion request returned HTTP {response.status_code}: {body[:1200]!r}")
        async for line in response.aiter_lines():
            if not line.startswith("data:"):
                continue
            data = line[5:].strip()
            if not data:
                continue
            if data == "[DONE]":
                break
            obj = json.loads(data)
            usage = obj.get("usage")
            if usage and usage.get("completion_tokens") is not None:
                usage_tokens = int(usage["completion_tokens"])
            choices = obj.get("choices") or []
            if not choices:
                continue
            piece = choices[0].get("text") or ""
            if not piece:
                continue
            now = time.perf_counter()
            if first_delta is None:
                first_delta = now
            if last_delta is not None:
                intervals.append(now - last_delta)
            last_delta = now
            pieces.append(piece)
    end = time.perf_counter()
    generated_text = "".join(pieces)
    token_count = usage_tokens if usage_tokens is not None else (
        len(tokenizer(generated_text, add_special_tokens=False)["input_ids"]) if generated_text else 0
    )
    decode_seconds = (end - first_delta) if first_delta is not None else None
    return {
        "ttft_seconds": round(first_delta - t0, 5) if first_delta is not None else None,
        "request_latency_seconds": round(end - t0, 5),
        "decode_seconds_after_first_token": round(decode_seconds, 5) if decode_seconds is not None else None,
        "output_tokens": token_count,
        "decode_tokens_per_second": round(max(0, token_count - 1) / decode_seconds, 4) if decode_seconds and decode_seconds > 0 else None,
        "mean_inter_chunk_latency_seconds": round(sum(intervals) / len(intervals), 5) if intervals else None,
        "stream_chunks_with_text": len(pieces),
        "generated_text": generated_text if include_text else None,
    }


async def run_load_mode(client, tokenizer, served_model):
    prompts = json.loads(LOAD_PROMPTS_PATH.read_text(encoding="utf-8"))
    prompts = prompts[:args.requests]
    # Warm up compilation and tokenizer paths outside the timed load interval.
    await stream_one(client, tokenizer, served_model, prompts[0], 8, False)
    start = time.perf_counter()
    results = await asyncio.gather(
        *[stream_one(client, tokenizer, served_model, prompt, 32, False) for prompt in prompts],
        return_exceptions=True,
    )
    wall = time.perf_counter() - start
    rows = []
    errors = []
    for i, item in enumerate(results):
        if isinstance(item, Exception):
            errors.append({"request_index": i, "error": repr(item)})
        else:
            rows.append(item)
    ttfts = [x["ttft_seconds"] for x in rows if x["ttft_seconds"] is not None]
    itls = [x["mean_inter_chunk_latency_seconds"] for x in rows if x["mean_inter_chunk_latency_seconds"] is not None]
    latencies = [x["request_latency_seconds"] for x in rows]
    tokens = sum(x["output_tokens"] for x in rows)
    return {
        "mode": "concurrent_load",
        "max_tokens_per_request": 32,
        "requested_concurrency": len(prompts),
        "completed_requests": len(rows),
        "failed_requests": len(errors),
        "errors": errors,
        "batch_wall_seconds": round(wall, 4),
        "output_tokens_total": tokens,
        "output_tokens_per_second": round(tokens / wall, 3) if wall else None,
        "mean_ttft_seconds": round(sum(ttfts) / len(ttfts), 5) if ttfts else None,
        "p50_ttft_seconds": percentile(ttfts, 0.5),
        "p95_ttft_seconds": percentile(ttfts, 0.95),
        "mean_inter_token_latency_estimate_seconds": round(sum(itls) / len(itls), 5) if itls else None,
        "p50_request_latency_seconds": percentile(latencies, 0.5),
        "p95_request_latency_seconds": percentile(latencies, 0.95),
    }


async def run_spec_mode(client, tokenizer, served_model):
    prompts = build_spec_prompts(tokenizer)
    # Warm up outside measurement with a shorter generation.
    await stream_one(client, tokenizer, served_model, prompts[0]["prompt"], 8, False)
    rows = []
    for i, item in enumerate(prompts):
        result = await stream_one(client, tokenizer, served_model, item["prompt"], 96, True)
        result["prompt_index"] = i
        result["prompt_tokens"] = item["prompt_tokens"]
        rows.append(result)
    tokens = sum(r["output_tokens"] for r in rows)
    decode_seconds = sum(r["decode_seconds_after_first_token"] or 0 for r in rows)
    ttfts = [r["ttft_seconds"] for r in rows if r["ttft_seconds"] is not None]
    return {
        "mode": "single_user_speculative_comparison",
        "requests": len(rows),
        "max_tokens_per_request": 96,
        "prompt_target_tokens": 100,
        "output_tokens_total": tokens,
        "aggregate_decode_tokens_per_second": round(max(0, tokens - len(rows)) / decode_seconds, 4) if decode_seconds else None,
        "mean_ttft_seconds": round(sum(ttfts) / len(ttfts), 5) if ttfts else None,
        "rows": rows,
    }


async def run_edge_checks(client, tokenizer, served_model):
    checks = {}
    # Empty input may be accepted or rejected with a client error; the key is no server crash.
    try:
        r = await client.post(f"{BASE_URL}/v1/completions", json={"model": served_model, "prompt": "", "max_tokens": 1, "temperature": 0, "stream": False})
        checks["empty_prompt"] = {"status_code": r.status_code, "handled": r.status_code < 500}
    except Exception as exc:
        checks["empty_prompt"] = {"handled": False, "error": repr(exc)}
    try:
        token_ids = tokenizer((" long-context-test" * 2500), add_special_tokens=False)["input_ids"][:2200]
        long_prompt = tokenizer.decode(token_ids, skip_special_tokens=True)
        r = await client.post(f"{BASE_URL}/v1/completions", json={"model": served_model, "prompt": long_prompt, "max_tokens": 1, "temperature": 0, "stream": False}, timeout=30.0)
        checks["over_context_prompt"] = {"input_tokens": len(token_ids), "status_code": r.status_code, "handled": r.status_code < 500}
    except Exception as exc:
        checks["over_context_prompt"] = {"handled": False, "error": repr(exc)}
    try:
        async with client.stream("POST", f"{BASE_URL}/v1/completions", json={"model": served_model, "prompt": "Write a long explanation about model inference.", "max_tokens": 64, "temperature": 0, "stream": True}) as response:
            async for line in response.aiter_lines():
                if line.startswith("data:") and line[5:].strip() not in ("", "[DONE]"):
                    break
        r = await client.get(f"{BASE_URL}/health")
        checks["early_client_disconnect"] = {"server_healthy_after_disconnect": r.status_code == 200}
    except Exception as exc:
        checks["early_client_disconnect"] = {"handled": False, "error": repr(exc)}
    try:
        r = await client.get(f"{BASE_URL}/health")
        checks["server_health_after_edge_tests"] = r.status_code == 200
    except Exception as exc:
        checks["server_health_after_edge_tests"] = {"error": repr(exc)}
    return checks


async def main():
    if args.speculative and not args.draft_model:
        raise ValueError("--draft-model is required for speculative mode")
    tokenizer = AutoTokenizer.from_pretrained(args.tokenizer, trust_remote_code=True)
    log_path = REPORT_DIR / f"vllm_server_{args.label}.log"
    output_path = Path(args.output)
    command = [
        VLLM_CLI, "serve", args.model,
        "--host", "127.0.0.1", "--port", str(args.port),
        "--dtype", "half", "--max-model-len", "2048",
        "--gpu-memory-utilization", "0.88", "--max-num-seqs", "32",
        "--trust-remote-code",
    ]
    spec_config = None
    if args.speculative:
        spec_config = {"method": "draft_model", "model": args.draft_model, "num_speculative_tokens": 5}
        command.extend(["--speculative-config", json.dumps(spec_config)])
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["HF_HUB_DISABLE_TELEMETRY"] = "1"
    env["LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + env.get("LIBRARY_PATH", "")
    env["LD_LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + env.get("LD_LIBRARY_PATH", "")
    memory_before = gpu_memory_rows()
    t0 = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, env=env)
        try:
            deadline = time.time() + 600
            ready = False
            while time.time() < deadline:
                code = proc.poll()
                if code is not None:
                    raise RuntimeError(f"vLLM exited early with code {code}. Log tail:\n{tail_text(log_path)}")
                try:
                    async with httpx.AsyncClient(timeout=3.0) as health_client:
                        response = await health_client.get(f"{BASE_URL}/health")
                        if response.status_code == 200:
                            ready = True
                            break
                except Exception:
                    pass
                await asyncio.sleep(2)
            if not ready:
                raise TimeoutError(f"vLLM did not become healthy within 600 seconds. Log tail:\n{tail_text(log_path)}")

            async with httpx.AsyncClient(timeout=httpx.Timeout(240.0, connect=30.0)) as client:
                model_resp = await client.get(f"{BASE_URL}/v1/models")
                model_resp.raise_for_status()
                served_model = model_resp.json()["data"][0]["id"]
                startup_seconds = time.perf_counter() - t0
                if args.mode == "load":
                    stage = await run_load_mode(client, tokenizer, served_model)
                    if args.label == "vllm_fp16_concurrency":
                        stage["edge_case_checks"] = await run_edge_checks(client, tokenizer, served_model)
                else:
                    stage = await run_spec_mode(client, tokenizer, served_model)
                memory_after = gpu_memory_rows()
                metrics_lines = []
                try:
                    metrics_response = await client.get(f"{BASE_URL}/metrics", timeout=10.0)
                    if metrics_response.status_code == 200:
                        metrics_lines = [line for line in metrics_response.text.splitlines() if any(term in line.lower() for term in ("kv_cache", "cache_usage", "num_requests_running", "num_requests_waiting"))][:100]
                except Exception:
                    pass
            result = {
                "label": args.label,
                "model_path": args.model,
                "tokenizer_path": args.tokenizer,
                "speculative_config": spec_config,
                "status": "ok" if stage.get("failed_requests", 0) == 0 else "partial",
                "engine": "vLLM OpenAI-compatible server",
                "vllm_version": "0.31.0 (pinned by notebook)",
                "hardware_visible_to_server": "Tesla T4 GPU 0 (compute capability 7.5)",
                "server_startup_seconds_including_model_load": round(startup_seconds, 2),
                "gpu_memory_before_server": memory_before,
                "gpu_memory_after_server_and_requests": memory_after,
                "kv_cache_and_memory_startup_log_lines": server_memory_lines(log_path),
                "kv_cache_prometheus_metrics": metrics_lines,
                "server_log": str(log_path),
                **stage,
            }
            output_path.parent.mkdir(parents=True, exist_ok=True)
            output_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
            print(json.dumps(result, indent=2))
        finally:
            if proc.poll() is None:
                proc.terminate()
                try:
                    proc.wait(timeout=25)
                except subprocess.TimeoutExpired:
                    proc.kill()
                    proc.wait(timeout=10)


try:
    asyncio.run(main())
except Exception as exc:
    result = {
        "label": args.label,
        "model_path": args.model,
        "mode": args.mode,
        "speculative": args.speculative,
        "status": "failed",
        "error": repr(exc),
        "traceback": traceback.format_exc(),
    }
    Path(args.output).parent.mkdir(parents=True, exist_ok=True)
    Path(args.output).write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps(result, indent=2))
    raise SystemExit(2)

## 8. Run 30 concurrent requests on FP16

This test uses vLLM's dynamic scheduler and paged KV-cache implementation. Pass criteria are measured: all 30 requests should complete, with aggregate throughput target of at least 150 tokens/s. The script also tests basic edge cases after the load test.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

ROOT = Path("/kaggle/working/llm-inference-artifacts-v2")
REPORT_DIR = ROOT / "reports"
BENCH_SCRIPT = Path("/tmp/run_vllm_bench_v2.py")
VLLM_PYTHON = "/kaggle/working/.venv-vllm-v2/bin/python"
BASE = ROOT / "qwen3-4b-original"
AWQ = ROOT / "qwen3-4b-awq-w4a16-symmetric"
DRAFT = ROOT / "qwen3-0.6b-draft"

def run_vllm_stage(label, model_path, mode, output_name, speculative=False):
    output = REPORT_DIR / output_name
    cmd = [
        VLLM_PYTHON, str(BENCH_SCRIPT),
        "--label", label,
        "--model", str(model_path),
        "--tokenizer", str(BASE),
        "--mode", mode,
        "--output", str(output),
        "--requests", "30",
    ]
    if speculative:
        cmd += ["--speculative", "--draft-model", str(DRAFT)]
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + env.get("LIBRARY_PATH", "")
    env["LD_LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + env.get("LD_LIBRARY_PATH", "")
    p = subprocess.run(cmd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout[-18000:])
    print("Exit code:", p.returncode)
    if output.exists():
        print("Saved result:", output)
        try:
            data = json.loads(output.read_text())
            print("Status:", data.get("status"))
        except Exception as exc:
            print("Result JSON could not be parsed:", exc)
    return p.returncode

assert BASE.is_dir(), f"Missing model directory: {BASE}. Run the quantization section first."
run_vllm_stage("vllm_fp16_concurrency", BASE, "load", "vllm_fp16_concurrency.json")

## 9. Run 30 concurrent requests on quantized AWQ

The new symmetric W4A16 checkpoint is served by the current isolated vLLM environment. If this stage fails, the result JSON and server log will preserve the actual compatibility error.

In [ ]:
assert AWQ.is_dir(), f"Missing symmetric AWQ checkpoint: {AWQ}. Inspect reports/results.json first."
run_vllm_stage("vllm_awq_concurrency", AWQ, "load", "vllm_awq_concurrency.json")

## 10. Measure target-only decoding

This is the baseline for speculative-decoding speedup. It uses single-user generation with approximately 100-token prompts and 96-token completions, not the high-concurrency test.

In [ ]:
run_vllm_stage("vllm_spec_target_only", BASE, "spec", "vllm_spec_target_only.json")

## 11. Measure draft-model speculative decoding

The same target is served with `Qwen3-0.6B` as the draft and a five-token speculation window. This cell records per-user decoding speed and generated text. The measured speedup and exact-output comparison are computed in the next section.

In [ ]:
assert DRAFT.is_dir(), f"Missing draft checkpoint: {DRAFT}. Run the draft download cell first."
run_vllm_stage("vllm_spec_draft", BASE, "spec", "vllm_spec_draft.json", speculative=True)

## 12. Consolidate acceptance metrics

This updates only files in `llm-inference-artifacts-v2`. It retains each raw per-stage JSON, server log, and the existing v1 artifacts.

In [ ]:
import json
from pathlib import Path

ROOT = Path("/kaggle/working/llm-inference-artifacts-v2")
REPORT_DIR = ROOT / "reports"
core_path = REPORT_DIR / "results.json"
summary_path = REPORT_DIR / "summary.md"
vllm_paths = {
    "fp16_concurrency": REPORT_DIR / "vllm_fp16_concurrency.json",
    "awq_concurrency": REPORT_DIR / "vllm_awq_concurrency.json",
    "spec_target_only": REPORT_DIR / "vllm_spec_target_only.json",
    "spec_draft": REPORT_DIR / "vllm_spec_draft.json",
}

def read(path):
    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else {"status": "missing", "error": f"Missing: {path}"}

core = read(core_path)
stages = {name: read(path) for name, path in vllm_paths.items()}
base = stages["spec_target_only"]
draft = stages["spec_draft"]

base_tps = base.get("aggregate_decode_tokens_per_second")
draft_tps = draft.get("aggregate_decode_tokens_per_second")
speedup = round(draft_tps / base_tps, 4) if base_tps and draft_tps else None
base_rows = base.get("rows", [])
draft_rows = draft.get("rows", [])
outputs_equal = (
    len(base_rows) == len(draft_rows) and len(base_rows) > 0 and
    all(a.get("generated_text") == b.get("generated_text") for a, b in zip(base_rows, draft_rows))
)

spec_validation = {
    "status": "measured" if speedup is not None else "not_available",
    "target_only_decode_tokens_per_second": base_tps,
    "draft_decode_tokens_per_second": draft_tps,
    "speedup_factor": speedup,
    "meets_1_4x_target": bool(speedup is not None and speedup >= 1.4),
    "exact_generated_text_match_all_prompts": outputs_equal if speedup is not None else None,
    "note": "This is a measured single-user comparison. Exact text equality is a practical greedy-decoding check, not a mathematical proof of equal distributions.",
}

fp16_load = stages["fp16_concurrency"]
awq_load = stages["awq_concurrency"]
hf_load = core.get("transformers_sequential_load_test", {}) if isinstance(core, dict) else {}

summary = {
    "overall_status": "partial",
    "model": core.get("model") if isinstance(core, dict) else None,
    "quantization": core.get("quantization") if isinstance(core, dict) else None,
    "quantization_results": {
        "original_model_size_gib": core.get("original_model_size_gib") if isinstance(core, dict) else None,
        "quantized_model_size_gib": core.get("quantized_model_size_gib") if isinstance(core, dict) else None,
        "size_reduction_percent": core.get("size_reduction_percent") if isinstance(core, dict) else None,
        "size_le_5_5_gib": core.get("quantized_model_size_pass_le_5_5_gib") if isinstance(core, dict) else None,
        "original_perplexity": core.get("original_perplexity", {}).get("perplexity") if isinstance(core, dict) and isinstance(core.get("original_perplexity"), dict) else None,
        "awq_perplexity": core.get("awq_perplexity", {}).get("perplexity") if isinstance(core, dict) and isinstance(core.get("awq_perplexity"), dict) else None,
        "perplexity_delta_percent": core.get("perplexity_delta_percent") if isinstance(core, dict) else None,
        "perplexity_within_5_percent": core.get("perplexity_within_5_percent") if isinstance(core, dict) else None,
    },
    "transformers_sequential_load": hf_load,
    "vllm_fp16_concurrency": fp16_load,
    "vllm_awq_concurrency": awq_load,
    "speculative_validation": spec_validation,
    "notes": [
        "vLLM uses its continuous scheduler and paged KV-cache implementation; this notebook measures 30 concurrent requests.",
        "Memory audit combines checkpoint size, vLLM model-load/KV-cache log lines, Prometheus KV-cache metrics, nvidia-smi snapshots, and Transformers peak allocations. These sources do not perfectly isolate every activation byte.",
        "FlashAttention-2 is not available on Tesla T4 (compute capability 7.5); the notebook runs a hardware check and records that limitation rather than claiming an FA2 benchmark.",
    ],
}

fp16_pass = fp16_load.get("status") == "ok" and fp16_load.get("completed_requests") == fp16_load.get("requested_concurrency") and (fp16_load.get("output_tokens_per_second") or 0) >= 150
awq_pass = awq_load.get("status") == "ok" and awq_load.get("completed_requests") == awq_load.get("requested_concurrency") and (awq_load.get("output_tokens_per_second") or 0) >= 150
ppl_pass = summary["quantization_results"].get("perplexity_within_5_percent") is True
size_pass = summary["quantization_results"].get("size_le_5_5_gib") is True
spec_pass = spec_validation["meets_1_4x_target"] and outputs_equal
summary["acceptance_checklist"] = {
    "awq_checkpoint_under_5_5_gib": bool(size_pass),
    "awq_perplexity_within_5_percent": bool(ppl_pass),
    "vllm_fp16_30_concurrent_requests_and_150_tps": bool(fp16_pass),
    "vllm_awq_30_concurrent_requests_and_150_tps": bool(awq_pass),
    "transformers_sequential_baseline_at_or_below_40_tps": (
        isinstance(hf_load.get("aggregate_tokens_per_second"), (int, float))
        and hf_load.get("aggregate_tokens_per_second") <= 40
    ),
    "draft_speculative_speedup_at_least_1_4x_and_outputs_match": bool(spec_pass),
    "flash_attention_2_on_t4": "hardware limitation; see kernel check",
}

statuses = [size_pass, ppl_pass, fp16_pass, awq_pass, spec_pass]
summary["overall_status"] = "ok" if all(statuses) else ("partial" if any(statuses) else "failed")
(REPORT_DIR / "vllm_results.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
if isinstance(core, dict) and "status" not in core:
    core["vllm_server_benchmarks"] = {"status": summary["overall_status"], **summary}
    core_path.write_text(json.dumps(core, indent=2), encoding="utf-8")

lines = [
    "# LLM Inference Optimization - v2 Acceptance Summary", "",
    f"- Overall status: `{summary['overall_status']}`", "",
    "## Quantization", "",
    f"- Original size: {summary['quantization_results']['original_model_size_gib']} GiB",
    f"- AWQ size: {summary['quantization_results']['quantized_model_size_gib']} GiB",
    f"- Reduction: {summary['quantization_results']['size_reduction_percent']}%",
    f"- AWQ size <= 5.5 GiB: `{size_pass}`",
    f"- FP16 perplexity: {summary['quantization_results']['original_perplexity']}",
    f"- AWQ perplexity: {summary['quantization_results']['awq_perplexity']}",
    f"- Perplexity delta: {summary['quantization_results']['perplexity_delta_percent']}%",
    f"- Perplexity within 5%: `{ppl_pass}`", "",
    "## Serving and speculative decoding", "",
    f"- Transformers sequential throughput: {hf_load.get('aggregate_tokens_per_second')} tokens/s (comparison target <= 40 tokens/s: `{summary['acceptance_checklist'].get('transformers_sequential_baseline_at_or_below_40_tps')}`)",
    f"- vLLM FP16 concurrency status/throughput: {fp16_load.get('status')} / {fp16_load.get('output_tokens_per_second')} tokens/s; completed {fp16_load.get('completed_requests')}/{fp16_load.get('requested_concurrency')}",
    f"- vLLM AWQ concurrency status/throughput: {awq_load.get('status')} / {awq_load.get('output_tokens_per_second')} tokens/s; completed {awq_load.get('completed_requests')}/{awq_load.get('requested_concurrency')}",
    f"- Draft speedup: {speedup}x; target >= 1.4x: `{spec_validation['meets_1_4x_target']}`",
    f"- Exact output match: `{outputs_equal}`", "",
    "## Acceptance checklist", "",
]
for key, value in summary["acceptance_checklist"].items():
    lines.append(f"- `{key}`: `{value}`")
lines += ["", "## Caveats", "", *[f"- {n}" for n in summary["notes"]], ""]
summary_path.write_text("\n".join(lines), encoding="utf-8")
print(json.dumps(summary["acceptance_checklist"], indent=2))
print("\nSpeculative validation:")
print(json.dumps(spec_validation, indent=2))
print("\nFull summary:", REPORT_DIR / "vllm_results.json")
print("Markdown summary:", summary_path)

## 13. Plot the measured throughput and latency

The plot reads the saved result files. Missing/failed stages are left out rather than assigned made-up values.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

REPORT_DIR = Path("/kaggle/working/llm-inference-artifacts-v2/reports")
def load_report(name):
    p = REPORT_DIR / name
    return json.loads(p.read_text()) if p.exists() else {}

results = [
    ("Transformers sequential", load_report("results.json").get("transformers_sequential_load_test", {}).get("aggregate_tokens_per_second")),
    ("vLLM FP16 30-way", load_report("vllm_fp16_concurrency.json").get("output_tokens_per_second")),
    ("vLLM AWQ 30-way", load_report("vllm_awq_concurrency.json").get("output_tokens_per_second")),
]
results = [(label, value) for label, value in results if isinstance(value, (int, float))]
if results:
    plt.figure(figsize=(8, 4))
    plt.bar([x[0] for x in results], [x[1] for x in results])
    plt.ylabel("Aggregate output tokens / second")
    plt.title("Sequential Transformers vs. concurrent vLLM")
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "throughput_comparison.png", dpi=160)
    plt.show()
else:
    print("No successful throughput measurements yet; run the benchmark cells first.")

base = load_report("vllm_spec_target_only.json")
draft = load_report("vllm_spec_draft.json")
if base.get("aggregate_decode_tokens_per_second") and draft.get("aggregate_decode_tokens_per_second"):
    plt.figure(figsize=(6, 4))
    plt.bar(
        ["Target only", "Target + draft"],
        [base["aggregate_decode_tokens_per_second"], draft["aggregate_decode_tokens_per_second"]],
    )
    plt.ylabel("Single-user decode tokens / second")
    plt.title("Speculative decoding speed comparison")
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "speculative_speed_comparison.png", dpi=160)
    plt.show()
else:
    print("Speculative comparison plot will be available once both single-user runs succeed.")

## 14. Attention-kernel benchmark: T4-compatible implementation

The upstream `flash-attn` package does not support Tesla T4 (SM75). The next cell optionally builds the community `flash-attention-turing` implementation in its own Python environment; it has a limited feature set and is not the upstream package. A build/runtime failure is recorded without stopping the remaining notebook. The existing PyTorch SDPA benchmark remains the fallback and is not labelled FlashAttention. [Turing fork README](https://github.com/ssiu/flash-attention-turing).


In [ ]:
%%writefile /tmp/benchmark_attention_v2.py
import json
import time
from pathlib import Path
import torch

report_dir = Path("/kaggle/working/llm-inference-artifacts-v2/reports")
report_dir.mkdir(parents=True, exist_ok=True)
if not torch.cuda.is_available():
    result = {"status": "failed", "error": "CUDA unavailable"}
else:
    cap = torch.cuda.get_device_capability(0)
    try:
        import flash_attn
        flash_installed = True
        flash_version = getattr(flash_attn, "__version__", "unknown")
    except Exception:
        flash_installed = False
        flash_version = None
    fa2_supported_arch = cap[0] >= 8
    result = {
        "gpu": torch.cuda.get_device_name(0),
        "compute_capability": list(cap),
        "flash_attn_package_installed": flash_installed,
        "flash_attn_version": flash_version,
        "flash_attention_2_benchmark_status": "not_run_hardware_unsupported" if not fa2_supported_arch else ("package_missing" if not flash_installed else "available"),
    }
    batch, heads, seq_len, head_dim = 2, 8, 512, 64
    q = torch.randn(batch, heads, seq_len, head_dim, device="cuda", dtype=torch.float16)
    k = torch.randn_like(q)
    v = torch.randn_like(q)
    fn = torch.nn.functional.scaled_dot_product_attention
    try:
        for _ in range(5):
            _ = fn(q, k, v, is_causal=True)
        torch.cuda.synchronize()
        times = []
        for _ in range(25):
            t0 = time.perf_counter()
            _ = fn(q, k, v, is_causal=True)
            torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
        result["pytorch_sdpa_fallback"] = {
            "status": "ok",
            "shape_batch_heads_sequence_head_dim": [batch, heads, seq_len, head_dim],
            "dtype": "float16",
            "repeats": len(times),
            "median_latency_ms": round(sorted(times)[len(times)//2] * 1000, 4),
            "mean_latency_ms": round(sum(times) / len(times) * 1000, 4),
            "peak_allocated_gpu_gib": round(torch.cuda.max_memory_allocated(0) / (1024**3), 4),
        }
    except Exception as exc:
        result["pytorch_sdpa_fallback"] = {"status": "failed", "error": repr(exc)}
    (report_dir / "attention_kernel_benchmark.json").write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))

In [ ]:
import os
import subprocess

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
p = subprocess.run(
    ["/kaggle/working/.venv-quant-v2/bin/python", "/tmp/benchmark_attention_v2.py"],
    env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print(p.stdout)
if p.returncode != 0:
    raise RuntimeError("Attention kernel diagnostic failed.")

## 15. Try a T4-specific FlashAttention kernel (isolated environment)

This experiment uses a separate PyTorch 2.8 / CUDA 12.8 environment, compiles a community implementation targeted at SM75, and compares its causal FP16 forward pass with PyTorch SDPA on identical tensors. If the driver/toolchain cannot build it, the failure is recorded and the main quantization/vLLM environments remain untouched.


In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/kaggle/working/llm-inference-artifacts-v2")
REPORT_DIR = ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_PATH = REPORT_DIR / "flash_attention_turing_benchmark.json"
FA_ENV = Path("/kaggle/working/.venv-fa-turing-v2")
FA_PY = FA_ENV / "bin/python"
FA_REPO = Path("/kaggle/working/flash-attention-turing")
steps = []
result = {
    "status": "failed",
    "implementation": "community flash-attention-turing fork for NVIDIA Turing/SM75",
    "upstream_flash_attn_package": False,
    "comparison": "same-shape FP16 causal forward pass vs PyTorch SDPA",
}

def run_step(label, command, cwd=None, env=None, timeout=1800):
    p = subprocess.run(
        [str(x) for x in command], cwd=str(cwd) if cwd else None, env=env,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout,
    )
    steps.append({"step": label, "returncode": p.returncode, "tail": p.stdout[-5000:]})
    print(f"--- {label}: exit {p.returncode} ---")
    print(p.stdout[-2500:])
    if p.returncode:
        raise RuntimeError(f"{label} failed with exit code {p.returncode}")
    return p

try:
    if not FA_PY.exists():
        run_step("create isolated Python 3.12 environment", ["uv", "venv", "--python", "3.12", "--seed", FA_ENV])
    run_step("install PyTorch 2.8 CUDA 12.8", ["uv", "pip", "install", "--python", FA_PY, "--torch-backend=cu128", "torch==2.8.0"])
    run_step("install build tools", ["uv", "pip", "install", "--python", FA_PY, "setuptools", "ninja", "wheel", "packaging"])
    if not (FA_REPO / ".git").is_dir():
        if FA_REPO.exists():
            shutil.rmtree(FA_REPO)
        run_step("clone Turing attention implementation", ["git", "clone", "--depth", "1", "https://github.com/ssiu/flash-attention-turing.git", FA_REPO], timeout=300)
    build_env = os.environ.copy()
    build_env["MAX_JOBS"] = "2"
    build_env["TORCH_CUDA_ARCH_LIST"] = "7.5"
    build_env["CUDA_HOME"] = "/usr/local/cuda"
    build_env["LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + build_env.get("LIBRARY_PATH", "")
    build_env["LD_LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + build_env.get("LD_LIBRARY_PATH", "")
    run_step("build/install the Turing attention extension", [FA_PY, "-m", "pip", "install", "-v", "."], cwd=FA_REPO, env=build_env, timeout=1800)

    bench_script = Path("/tmp/benchmark_flash_attention_turing_v2.py")
    bench_script.write_text(r"""import json, time, statistics, traceback
from pathlib import Path
import torch

out_path = Path("/kaggle/working/llm-inference-artifacts-v2/reports/flash_attention_turing_benchmark.json")
try:
    from flash_attention_interface import flash_attn_func
    assert torch.cuda.is_available(), "CUDA is not available"
    cap = torch.cuda.get_device_capability(0)
    assert cap == (7, 5), f"Expected T4 SM75, got {cap}"
    torch.manual_seed(7)
    batch, seq, heads, dim = 2, 512, 8, 64
    q = torch.randn(batch, seq, heads, dim, device="cuda", dtype=torch.float16)
    k = torch.randn_like(q)
    v = torch.randn_like(q)
    def flash_fn():
        return flash_attn_func(q, k, v, causal=True)
    def sdpa_fn():
        return torch.nn.functional.scaled_dot_product_attention(
            q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), is_causal=True
        ).transpose(1, 2).contiguous()
    with torch.inference_mode():
        flash_out = flash_fn()
        sdpa_out = sdpa_fn()
        torch.cuda.synchronize()
        error = float((flash_out.float() - sdpa_out.float()).abs().max().cpu())
        def measure(fn, n=25):
            for _ in range(5):
                fn()
            torch.cuda.synchronize()
            vals = []
            for _ in range(n):
                t = time.perf_counter()
                fn()
                torch.cuda.synchronize()
                vals.append(time.perf_counter() - t)
            return {"median_latency_ms": round(statistics.median(vals) * 1000, 4), "mean_latency_ms": round(statistics.mean(vals) * 1000, 4), "repeats": n}
        torch.cuda.reset_peak_memory_stats(0)
        flash = measure(flash_fn)
        sdpa = measure(sdpa_fn)
        result = {
            "status": "ok", "implementation": "community flash-attention-turing fork",
            "gpu": torch.cuda.get_device_name(0), "compute_capability": list(cap),
            "shape_batch_sequence_heads_head_dim": [batch, seq, heads, dim],
            "dtype": "float16", "causal": True,
            "flash_attention_turing": flash, "pytorch_sdpa": sdpa,
            "speedup_vs_sdpa_median": round(sdpa["median_latency_ms"] / flash["median_latency_ms"], 4),
            "max_abs_difference_vs_sdpa": round(error, 6),
            "peak_allocated_gpu_gib": round(torch.cuda.max_memory_allocated(0) / (1024**3), 4),
            "note": "Standalone causal forward-pass microbenchmark; it does not imply the same speedup inside the full LLM-serving stack."
        }
except Exception as exc:
    result = {"status": "failed", "error": repr(exc), "traceback": traceback.format_exc()}
out_path.write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))
""", encoding="utf-8")
    bench_env = os.environ.copy()
    bench_env["CUDA_VISIBLE_DEVICES"] = "0"
    run_step("benchmark Turing attention vs SDPA", [FA_PY, bench_script], env=bench_env, timeout=300)
    result.update(json.loads(RESULT_PATH.read_text(encoding="utf-8")))
except Exception as exc:
    result.update({"status": "failed", "error": repr(exc)})
finally:
    result["installation_steps"] = steps
    RESULT_PATH.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print("Turing attention experiment status:", result.get("status"))
    print("Report:", RESULT_PATH)


## 15. Final verification and artifact checklist

The main outputs stay under `/kaggle/working/llm-inference-artifacts-v2/`. Review the acceptance checklist in `reports/vllm_results.json` and `reports/summary.md`. Keep the raw per-stage JSONs and logs for any failed test so the next iteration targets the real cause.

In [ ]:
import json
from pathlib import Path

ROOT = Path("/kaggle/working/llm-inference-artifacts-v2")
REPORTS = ROOT / "reports"
for name in [
    "results.json",
    "transformers_benchmarks.csv",
    "summary.md",
    "load_prompts.json",
    "vllm_fp16_concurrency.json",
    "vllm_awq_concurrency.json",
    "vllm_spec_target_only.json",
    "vllm_spec_draft.json",
    "vllm_results.json",
    "attention_kernel_benchmark.json",
    "flash_attention_turing_benchmark.json",
    "throughput_comparison.png",
    "speculative_speed_comparison.png",
]:
    p = REPORTS / name
    print(f"{'OK' if p.exists() else 'not created yet'}  {p}")

summary = REPORTS / "vllm_results.json"
if summary.exists():
    data = json.loads(summary.read_text())
    print("\nAcceptance checklist:")
    print(json.dumps(data.get("acceptance_checklist", {}), indent=2))